In [18]:
# Install FastAPI, Uvicorn, and nest_asyncio
!pip install -q fastapi uvicorn nest_asyncio

/usr/lib/python3.13/pathlib/_local.py:274: RuntimeWarning: coroutine 'Server.serve' was never awaited
  parsed = [sys.intern(str(x)) for x in rel.split(sep) if x and x != '.']


In [19]:
from fastapi import FastAPI, Request
from fastapi.responses import HTMLResponse
from pydantic import BaseModel
import pandas as pd
import numpy as np
from datetime import date

# Assuming these data structures are defined elsewhere or loaded from files.
# For demonstration, I'll define placeholders or re-initialize them based on context
# If they exist in kernel state, they will be used.

# --- Placeholder/Re-initialization for demonstration ---
# In a real scenario, these would be loaded from a database, JSON, or CSV.
# Given the kernel state, I will assume these DataFrames/dicts are already populated.
# However, to make the app executable independently, I'll include minimal definitions.

# Mock data for INGREDIENTS_DB, PLAN_MATRIX, REGIONAL_PRICING, RECIPES_DB, SPLIT_RULES
# Based on kernel state, these are already present. The following are minimal to make the code run.

if 'INGREDIENTS_DB' not in locals():
    INGREDIENTS_DB = {
        1: {'id': 1, 'name_en': 'Chicken Breast (Raw)', 'name_ar': 'صدر دجاج (نيء)', 'cal_per_g': 1.2, 'pro_per_g': 0.225, 'carb_per_g': 0.0, 'fat_per_g': 0.026},
        2: {'id': 2, 'name_en': 'Basmati White Rice (Raw)', 'name_ar': 'أرز بسمتي أبيض (غير مطبوخ)', 'cal_per_g': 3.6, 'pro_per_g': 0.07, 'carb_per_g': 0.8, 'fat_per_g': 0.007}
    }

if 'RECIPES_DB' not in locals():
    RECIPES_DB = [
        {'id': 101, 'slot': 'lunch', 'name_en': 'Spiced Arabic Chicken & Basmati Rice', 'name_ar': 'دجاج متبل مع أرز بسمتي', 'method_en': 'Cook chicken and rice.', 'method_ar': 'اطبخ دجاج وأرز.', 'ingredients': [{'id': 1, 'base_g': 100}, {'id': 2, 'base_g': 70}], 'base_cal': 500, 'base_pro': 50, 'base_carb': 60, 'base_fat': 15},
        {'id': 102, 'slot': 'breakfast', 'name_en': 'High-Protein Berry Oatmeal Bowl', 'name_ar': 'وعاء الشوفان والتوت عالي البروتين', 'method_en': 'Cook oats and add berries.', 'method_ar': 'اطبخ الشوفان وأضف التوت.', 'ingredients': [{'id': 5, 'base_g': 100}, {'id': 6, 'base_g': 50}], 'base_cal': 300, 'base_pro': 20, 'carb_per_g': 40, 'fat_per_g': 10}
    ]

if 'PLAN_MATRIX' not in locals():
    PLAN_MATRIX = [
        {'plan_type': 'weight_loss', 'calories': 1400, 'carbs_g': 157.5, 'protein_g': 105.0, 'fat_g': 38.9},
        {'plan_type': 'standard', 'calories': 2000, 'carbs_g': 250.0, 'protein_g': 125.0, 'fat_g': 55.6}
    ]

if 'REGIONAL_PRICING' not in locals():
    REGIONAL_PRICING = {
        'JO': {'currency': 'JOD', 1: 0.0045, 2: 0.0015},
        'SA': {'currency': 'SAR', 1: 0.028, 2: 0.009}
    }

if 'SPLIT_RULES' not in locals():
    SPLIT_RULES = {
        'b_l_d': {'breakfast': 0.3, 'lunch': 0.4, 'dinner': 0.3}
    }

# HTML_UI must be defined, as it's used in the root endpoint
if 'HTML_UI' not in locals():
    HTML_UI = """<!DOCTYPE html><html><head><title>Munch Me Engine</title></head><body><h1>Welcome to Munch Me!</h1><div id="app-root"></div><script>document.getElementById('app-root').innerText = 'Loading frontend...';</script></body></html>"""

# Helper functions (simplified for brevity, actual logic from original notebook)
# user_profile is in kernel state, assuming it's already defined.

def calculate_bmr(user_profile):
    # Simplified BMR calculation
    age = (date.today() - user_profile['dob']).days / 365
    if user_profile['sex'] == 'm':
        return (10 * user_profile['weight_kg']) + (6.25 * user_profile['height_cm']) - (5 * age) + 5
    else:
        return (10 * user_profile['weight_kg']) + (6.25 * user_profile['height_cm']) - (5 * age) - 161

def get_plan(goal, calories):
    plan_df = pd.DataFrame(PLAN_MATRIX)
    filtered_plan = plan_df[(plan_df['plan_type'] == goal) & (plan_df['calories'] == calories)]
    if not filtered_plan.empty:
        return filtered_plan.iloc[0].to_dict()
    return None

def scale_recipe(recipe, target_calories, ingredients_db):
    # This is a simplified scaling. Original logic is more complex.
    # Assume recipe['base_cal'] and other 'base_' values exist.
    if recipe['base_cal'] == 0: # Avoid division by zero
        scale_factor = 1
    else:
        scale_factor = target_calories / recipe['base_cal']

    scaled_ingredients = []
    for ing in recipe['ingredients']:
        scaled_g = ing['base_g'] * scale_factor
        ingredient_details = ingredients_db.get(ing['id'], {})
        scaled_ingredients.append({
            'id': ing['id'],
            'name_en': ingredient_details.get('name_en', 'Unknown'),
            'scaled_g': scaled_g,
            'scaled_cal': scaled_g * ingredient_details.get('cal_per_g', 0),
            'scaled_pro': scaled_g * ingredient_details.get('pro_per_g', 0),
            'scaled_carb': scaled_g * ingredient_details.get('carb_per_g', 0),
            'scaled_fat': scaled_g * ingredient_details.get('fat_per_g', 0),
        })

    scaled_recipe = recipe.copy()
    scaled_recipe['scaled_calories'] = recipe['base_cal'] * scale_factor
    scaled_recipe['scaled_protein'] = recipe['base_pro'] * scale_factor
    scaled_recipe['scaled_carbs'] = recipe['base_carb'] * scale_factor
    scaled_recipe['scaled_fat'] = recipe['base_fat'] * scale_factor
    scaled_recipe['scaled_ingredients'] = scaled_ingredients
    return scaled_recipe


class UserProfile(BaseModel):
    name: str
    dob: date
    sex: str
    height_cm: float
    weight_kg: float
    pal_factor: float
    goal: str
    country: str
    meal_config: list[str]

class PlanRequest(BaseModel):
    user_profile: UserProfile

app = FastAPI()

@app.get("/", response_class=HTMLResponse)
async def read_root():
    return HTML_UI

@app.post("/generate_meal_plan")
async def generate_meal_plan(request: PlanRequest):
    user_profile_data = request.user_profile.dict()
    user_profile_data['dob'] = user_profile_data['dob'].isoformat() # Convert date to string for calculation consistency if needed

    # Calculate BMR and TDEE
    bmr = calculate_bmr(request.user_profile)
    tdee = bmr * request.user_profile.pal_factor

    # Get target calories and macros based on goal
    target_plan = get_plan(request.user_profile.goal, user_profile['target_calories'] if 'target_calories' in user_profile else 1800) # Use a default or value from kernel state

    if not target_plan:
        return {"error": "Could not find a meal plan for the given goal and calories."}

    # Distribute calories by meal configuration
    meal_plan = {}
    total_target_calories = target_plan['calories']

    # This part needs to be more robust, using SPLIT_RULES data.
    # For simplicity, using a basic distribution for now.
    split_rule_key = '_'.join([m[0] for m in request.user_profile.meal_config]) # e.g., b_l_d
    meal_splits = SPLIT_RULES.get(split_rule_key, {'breakfast': 0.3, 'lunch': 0.4, 'dinner': 0.3}) # Default if rule not found

    available_recipes = pd.DataFrame(RECIPES_DB)

    for meal_slot, proportion in meal_splits.items():
        target_kcal = total_target_calories * proportion

        # Filter recipes for the current meal slot
        slot_recipes = available_recipes[available_recipes['slot'] == meal_slot]

        if not slot_recipes.empty:
            # For simplicity, pick the first recipe that loosely fits, or the one with base_cal closest to target
            # A more sophisticated approach would involve optimization or more selection logic
            selected_recipe = slot_recipes.iloc[0].to_dict() # Just taking the first for now
            scaled = scale_recipe(selected_recipe, target_kcal, INGREDIENTS_DB)
            meal_plan[meal_slot] = scaled
        else:
            meal_plan[meal_slot] = {"error": f"No recipes found for {meal_slot}"}

    # Calculate grocery list and pricing
    grocery_list = {}
    total_cost = 0
    currency = REGIONAL_PRICING.get(request.user_profile.country, {}).get('currency', 'USD')

    for meal_slot, recipe_data in meal_plan.items():
        if 'scaled_ingredients' in recipe_data:
            for ing in recipe_data['scaled_ingredients']:
                ing_id = ing['id']
                scaled_g = ing['scaled_g']

                # Get pricing for the ingredient in the user's country
                cost_per_g = REGIONAL_PRICING.get(request.user_profile.country, {}).get(ing_id, 0)
                ingredient_cost = scaled_g * cost_per_g
                total_cost += ingredient_cost

                if ing_id not in grocery_list:
                    grocery_list[ing_id] = {
                        'name_en': ing['name_en'],
                        'total_g': 0,
                        'total_cost': 0
                    }
                grocery_list[ing_id]['total_g'] += scaled_g
                grocery_list[ing_id]['total_cost'] += ingredient_cost

    formatted_grocery_list = [
        {
            'name_en': item['name_en'],
            'amount_g': round(item['total_g'], 2),
            'cost': f"{round(item['total_cost'], 2)} {currency}"
        } for ing_id, item in grocery_list.items()
    ]

    return {
        "user_profile": user_profile_data,
        "bmr": round(bmr, 2),
        "tdee": round(tdee, 2),
        "target_plan": target_plan,
        "meal_plan": meal_plan,
        "grocery_list": formatted_grocery_list,
        "total_grocery_cost": f"{round(total_cost, 2)} {currency}"
    }

In [20]:
import nest_asyncio
import uvicorn
import threading
from google.colab import output

nest_asyncio.apply()

# Run the server quietly in the background without blocking Colab
def start_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning") # Changed to 0.0.0.0 for external access

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()

print("✅ Server running in background!")
print("Opening the Munch Me web interface now...")

# Colab's built-in tool to view the page without any accounts or tokens
# output.serve_kernel_port_as_window(8000) # Re-enabling this as it's the intended way to view the UI
# Using serve_kernel_port_as_iframe as suggested by the warning
output.serve_kernel_port_as_iframe(8000)

✅ Server running in background!
Opening the Munch Me web interface now...


<IPython.core.display.Javascript object>

ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use


In [21]:
# This cell was an earlier attempt to run the server which failed. It is now redundant.
# The server will be run by the '0rQcQUOc9NAt' cell (which has been modified for nest_asyncio and threading).
# Therefore, this cell can be safely commented out or deleted if desired.
# if __name__ == "__main__":
#     print("\n🚀 Munch Me Application Starting at: http://localhost:8000")
#     uvicorn.run(app, host="0.0.0.0", port=8000)

In [22]:
# This cell was an earlier attempt to run the server which failed with an event loop error.
# The server is now handled by cell `ce82762e` with proper `nest_asyncio` setup and background threading.
# This cell should be ignored or deleted.
# if __name__ == "__main__":
#     print("\n🚀 Munch Me Application Starting at: http://localhost:8000")
#     uvicorn.run(app, host="0.0.0.0", port=8000)

In [23]:
import threading
import uvicorn
from google.colab import output

# This cell previously started a uvicorn server.
# The server is now handled by a later cell (ce82762e) with proper nest_asyncio setup.
# Commenting out to avoid 'address already in use' errors and ensure correct server launch.

# # Run the server quietly in the background without blocking Colab
# def start_server():
#     uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

# server_thread = threading.Thread(target=start_server, daemon=True)
# server_thread.start()

# print("✅ Server running in background!")
# print("Opening the Munch Me web interface now...")

# Colab's built-in tool to view the page without any accounts or tokens
# output.serve_kernel_port_as_window(8000)

In [24]:
from IPython.display import HTML, display

# Embed the interactive UI directly inside your Colab output
display(
    HTML("""
<div style="border: 1px solid #e2e8f0; border-radius: 12px; overflow: hidden; max-width: 100%;">
  <iframe srcdoc='"""
        + HTML_UI.replace("'", "&#39;")
        + """'
          width="100%"
          height="850px"
          frameborder="0"
          style="background: #f8fafc;">
  </iframe>
</div>
""")
)

Pantry,Ingredient,Total Weight,Regional Cost


In [25]:
from IPython.display import HTML, display

# ==============================================================================
# 1. EXPANDED MASTER INGREDIENT 1G REFERENCE MATRIX
# ==============================================================================
INGREDIENTS_DB_V2 = {
    # Proteins
    1: {
        "id": 1,
        "name_en": "Chicken Breast (Raw)",
        "name_ar": "صدر دجاج (نيء)",
        "dept_en": "Meat & Poultry",
        "dept_ar": "لحوم ودواجن",
        "cal_per_g": 1.20,
        "pro_per_g": 0.225,
        "carb_per_g": 0.0,
        "fat_per_g": 0.026,
    },
    2: {
        "id": 2,
        "name_en": "Lean Ground Beef (90/10)",
        "name_ar": "لحم بقر مفروم قليل الدهن",
        "dept_en": "Meat & Poultry",
        "dept_ar": "لحوم ودواجن",
        "cal_per_g": 1.76,
        "pro_per_g": 0.200,
        "carb_per_g": 0.0,
        "fat_per_g": 0.100,
    },
    3: {
        "id": 3,
        "name_en": "Whole Eggs",
        "name_ar": "بيض طازج",
        "dept_en": "Dairy & Eggs",
        "dept_ar": "ألبان وبيض",
        "cal_per_g": 1.43,
        "pro_per_g": 0.126,
        "carb_per_g": 0.008,
        "fat_per_g": 0.095,
    },
    4: {
        "id": 4,
        "name_en": "Low-Fat Halloumi Cheese",
        "name_ar": "جبنة حلوم قليلة الدسم",
        "dept_en": "Dairy & Eggs",
        "dept_ar": "ألبان وبيض",
        "cal_per_g": 2.60,
        "pro_per_g": 0.220,
        "carb_per_g": 0.020,
        "fat_per_g": 0.180,
    },
    5: {
        "id": 5,
        "name_en": "Traditional Labneh (Low-Fat)",
        "name_ar": "لبنة بلدية قليلة الدسم",
        "dept_en": "Dairy & Eggs",
        "dept_ar": "ألبان وبيض",
        "cal_per_g": 1.05,
        "pro_per_g": 0.090,
        "carb_per_g": 0.040,
        "fat_per_g": 0.055,
    },
    6: {
        "id": 6,
        "name_en": "Greek Yogurt (0% Fat)",
        "name_ar": "لبن زبادي يوناني (خالي الدسم)",
        "dept_en": "Dairy & Eggs",
        "dept_ar": "ألبان وبيض",
        "cal_per_g": 0.59,
        "pro_per_g": 0.100,
        "carb_per_g": 0.036,
        "fat_per_g": 0.004,
    },
    # Grains & Starches
    7: {
        "id": 7,
        "name_en": "Basmati White Rice (Raw)",
        "name_ar": "أرز بسمتي أبيض (غير مطبوخ)",
        "dept_en": "Pantry & Grains",
        "dept_ar": "حبوب وبقوليات",
        "cal_per_g": 3.60,
        "pro_per_g": 0.070,
        "carb_per_g": 0.800,
        "fat_per_g": 0.007,
    },
    8: {
        "id": 8,
        "name_en": "Whole Green Freekeh (Dry)",
        "name_ar": "فريكة خضراء حب (جافة)",
        "dept_en": "Pantry & Grains",
        "dept_ar": "حبوب وبقوليات",
        "cal_per_g": 3.25,
        "pro_per_g": 0.145,
        "carb_per_g": 0.650,
        "fat_per_g": 0.022,
    },
    9: {
        "id": 9,
        "name_en": "Quinoa (Dry)",
        "name_ar": "كينوا (جافة)",
        "dept_en": "Pantry & Grains",
        "dept_ar": "حبوب وبقوليات",
        "cal_per_g": 3.68,
        "pro_per_g": 0.141,
        "carb_per_g": 0.642,
        "fat_per_g": 0.061,
    },
    10: {
        "id": 10,
        "name_en": "Rolled Oats",
        "name_ar": "شوفان حبة كاملة",
        "dept_en": "Pantry & Grains",
        "dept_ar": "حبوب وبقوليات",
        "cal_per_g": 3.89,
        "pro_per_g": 0.169,
        "carb_per_g": 0.663,
        "fat_per_g": 0.069,
    },
    11: {
        "id": 11,
        "name_en": "Whole Wheat Sourdough Bread",
        "name_ar": "خبز قمح كامل مخمر",
        "dept_en": "Bakery",
        "dept_ar": "مخبوزات",
        "cal_per_g": 2.38,
        "pro_per_g": 0.110,
        "carb_per_g": 0.440,
        "fat_per_g": 0.020,
    },
    12: {
        "id": 12,
        "name_en": "Sweet Potato (Raw)",
        "name_ar": "بطاطا حلوة نيئة",
        "dept_en": "Produce",
        "dept_ar": "خضار وفواكه",
        "cal_per_g": 0.86,
        "pro_per_g": 0.016,
        "carb_per_g": 0.201,
        "fat_per_g": 0.001,
    },
    # Fats & Oils
    13: {
        "id": 13,
        "name_en": "Extra Virgin Olive Oil",
        "name_ar": "زيت زيتون بكر ممتاز",
        "dept_en": "Pantry & Grains",
        "dept_ar": "حبوب وبقوليات",
        "cal_per_g": 8.84,
        "pro_per_g": 0.0,
        "carb_per_g": 0.0,
        "fat_per_g": 1.000,
    },
    14: {
        "id": 14,
        "name_en": "Tahini Paste (100% Sesame)",
        "name_ar": "طحينية سمسم نقية",
        "dept_en": "Pantry & Grains",
        "dept_ar": "حبوب وبقوليات",
        "cal_per_g": 5.95,
        "pro_per_g": 0.170,
        "carb_per_g": 0.210,
        "fat_per_g": 0.540,
    },
    15: {
        "id": 15,
        "name_en": "Raw Almonds",
        "name_ar": "لوز نيء",
        "dept_en": "Pantry & Grains",
        "dept_ar": "حبوب وبقوليات",
        "cal_per_g": 5.79,
        "pro_per_g": 0.212,
        "carb_per_g": 0.216,
        "fat_per_g": 0.499,
    },
    # Produce
    16: {
        "id": 16,
        "name_en": "Fresh Cucumber & Mint",
        "name_ar": "خيار ونعناع طازج",
        "dept_en": "Produce",
        "dept_ar": "خضار وفواكه",
        "cal_per_g": 0.15,
        "pro_per_g": 0.007,
        "carb_per_g": 0.036,
        "fat_per_g": 0.001,
    },
    17: {
        "id": 17,
        "name_en": "Baby Spinach & Arugula",
        "name_ar": "سبانخ وجرجير طازج",
        "dept_en": "Produce",
        "dept_ar": "خضار وفواكه",
        "cal_per_g": 0.23,
        "pro_per_g": 0.029,
        "carb_per_g": 0.036,
        "fat_per_g": 0.004,
    },
    18: {
        "id": 18,
        "name_en": "Fresh Mixed Berries",
        "name_ar": "توت مشكل طازج",
        "dept_en": "Produce",
        "dept_ar": "خضار وفواكه",
        "cal_per_g": 0.57,
        "pro_per_g": 0.007,
        "carb_per_g": 0.140,
        "fat_per_g": 0.003,
    },
    # Spices (Itemized)
    19: {
        "id": 19,
        "name_en": "Shawarma Spice Blend & Garlic",
        "name_ar": "بهارات شاورما وثوم مهروس",
        "dept_en": "Spices & Seasoning",
        "dept_ar": "بهارات وتوابل",
        "cal_per_g": 0.0,
        "pro_per_g": 0.0,
        "carb_per_g": 0.0,
        "fat_per_g": 0.0,
    },
    20: {
        "id": 20,
        "name_en": "Za'atar & Sumac Blend",
        "name_ar": "خلطة زعتر بلدي وسماق",
        "dept_en": "Spices & Seasoning",
        "dept_ar": "بهارات وتوابل",
        "cal_per_g": 0.0,
        "pro_per_g": 0.0,
        "carb_per_g": 0.0,
        "fat_per_g": 0.0,
    },
}

# ==============================================================================
# 2. LOCALIZED INGREDIENT PRICING (Per 1g)
# ==============================================================================
REGIONAL_PRICING_V2 = {
    "JO": {
        "currency": "JOD",
        1: 0.0045,
        2: 0.0075,
        3: 0.0035,
        4: 0.0080,
        5: 0.0040,
        6: 0.0050,
        7: 0.0015,
        8: 0.0022,
        9: 0.0040,
        10: 0.0030,
        11: 0.0025,
        12: 0.0012,
        13: 0.0090,
        14: 0.0055,
        15: 0.0120,
        16: 0.0008,
        17: 0.0010,
        18: 0.0080,
        19: 0.0020,
        20: 0.0020,
    },
    "SA": {
        "currency": "SAR",
        1: 0.0280,
        2: 0.0420,
        3: 0.0180,
        4: 0.0480,
        5: 0.0240,
        6: 0.0220,
        7: 0.0090,
        8: 0.0140,
        9: 0.0240,
        10: 0.0150,
        11: 0.0120,
        12: 0.0065,
        13: 0.0450,
        14: 0.0280,
        15: 0.0650,
        16: 0.0045,
        17: 0.0055,
        18: 0.0400,
        19: 0.0080,
        20: 0.0080,
    },
    "AE": {
        "currency": "AED",
        1: 0.0270,
        2: 0.0400,
        3: 0.0175,
        4: 0.0460,
        5: 0.0230,
        6: 0.0210,
        7: 0.0085,
        8: 0.0135,
        9: 0.0230,
        10: 0.0140,
        11: 0.0115,
        12: 0.0060,
        13: 0.0440,
        14: 0.0270,
        15: 0.0620,
        16: 0.0040,
        17: 0.0050,
        18: 0.0380,
        19: 0.0075,
        20: 0.0075,
    },
}

# ==============================================================================
# 3. RECIPES (Pure methods, Itemized spices, Gram baselines)
# ==============================================================================
RECIPES_DB_V2 = [
    # Breakfast Options
    {
        "id": 101,
        "slot": "breakfast",
        "name_en": "Za'atar Labneh & Whole Wheat Sourdough",
        "name_ar": "توست اللبنة البلدية بالزعتر وزيت الزيتون",
        "method_en": "1. Lightly toast the sourdough bread until crisp.\n2. Spread chilled labneh evenly over the surface.\n3. Drizzle olive oil and dust with za'atar sumac blend.\n4. Serve fresh with cucumber and mint.",
        "method_ar": "١. تحميص خبز القمح الكامل حتى يصبح مقرمشاً.\n٢. فرد اللبنة البلدية الباردة بالتساوي على الوجه.\n٣. سكب زيت الزيتون ورش خلطة الزعتر والسماق.\n٤. يُقدم مباشرة مع الخيار والنعناع الطازج.",
        "ingredients": [
            {"id": 11, "base_g": 60},
            {"id": 5, "base_g": 90},
            {"id": 13, "base_g": 5},
            {"id": 20, "base_g": 3},
            {"id": 16, "base_g": 50},
        ],
    },
    {
        "id": 102,
        "slot": "breakfast",
        "name_en": "Golden Sweet Potato & Egg Skillet",
        "name_ar": "مقلاة البطاطا الحلوة والبيض المشوي",
        "method_en": "1. Dice sweet potato and sauté in olive oil until soft and caramelized.\n2. Whisk whole eggs and pour over the potatoes.\n3. Cook gently over low heat until eggs are softly set.\n4. Serve warm.",
        "method_ar": "١. تقطيع البطاطا الحلوة وتشويحها بزيت الزيتون حتى تلين وتتحمر.\n٢. خفق البيض الطازج وسكبه فوق البطاطا.\n٣. الطهي على نار هادئة حتى ينضج البيض تماماً.\n٤. يُقدم الطبق ساخناً.",
        "ingredients": [
            {"id": 12, "base_g": 120},
            {"id": 3, "base_g": 110},
            {"id": 13, "base_g": 6},
        ],
    },
    # Lunch Options
    {
        "id": 201,
        "slot": "lunch",
        "name_en": "Healthy Chicken Shawarma Plate with Basmati",
        "name_ar": "طبق شاورما الدجاج الصحية مع أرز بسمتي",
        "method_en": "1. Rub raw chicken strips with shawarma spices.\n2. Sear chicken in hot olive oil until thoroughly cooked and slightly charred.\n3. Steam basmati rice.\n4. Whisk tahini with lemon juice and a splash of water for the dressing.\n5. Assemble plate with warm rice, spiced chicken, fresh greens, and tahini drizzle.",
        "method_ar": "١. تتبيل شرائح الدجاج بخلطة بهارات الشاورما والثوم.\n٢. تشويح الدجاج في مقلاة ساخنة بزيت الزيتون حتى يكتسب لون التحمير وينضج.\n٣. طهي أرز البسمتي على البخار.\n٤. خفق الطحينية مع قليل من الماء وعصير الليمون.\n٥. ترتيب الأرز، الدجاج المحمر، الورقيات، وسكب صلصة الطحينية على الوجه.",
        "ingredients": [
            {"id": 1, "base_g": 160},
            {"id": 7, "base_g": 60},
            {"id": 13, "base_g": 6},
            {"id": 14, "base_g": 12},
            {"id": 19, "base_g": 4},
            {"id": 17, "base_g": 30},
        ],
    },
    {
        "id": 202,
        "slot": "lunch",
        "name_en": "Spiced Green Freekeh with Lean Beef",
        "name_ar": "فريكة خضراء باللحم البقري المفروم والمكسرات",
        "method_en": "1. Rinse freekeh thoroughly; simmer in water or broth until tender and smoky.\n2. Brown lean ground beef in a skillet with olive oil and spices.\n3. Layer cooked freekeh into a serving bowl, top with seasoned beef.\n4. Garnish with almonds.",
        "method_ar": "١. غسل الفريكة الخضراء وطهيها بالماء المغلي حتى تنضج وتمتص النكهة المدخنة.\n٢. طهي اللحم البقري المفروم بزيت الزيتون والبهارات حتى ينضج تماماً.\n٣. سكب الفريكة في طبق التقديم ووضع اللحم المفروم فوقها.\n٤. التزيين بحبات اللوز وتقديمها ساخنة.",
        "ingredients": [
            {"id": 8, "base_g": 65},
            {"id": 2, "base_g": 130},
            {"id": 13, "base_g": 6},
            {"id": 15, "base_g": 12},
            {"id": 19, "base_g": 3},
        ],
    },
    # Dinner Options
    {
        "id": 301,
        "slot": "dinner",
        "name_en": "Grilled Halloumi & Herb Quinoa Bowl",
        "name_ar": "وعاء الكينوا بجبنة الحلوم المشوية والنعناع",
        "method_en": "1. Simmer quinoa in water until fluffy, then allow to cool slightly.\n2. Sear halloumi cheese slices in a dry non-stick skillet until golden crust forms.\n3. Toss quinoa with fresh spinach, arugula, diced cucumber, and olive oil.\n4. Top with warm grilled halloumi and dust with sumac.",
        "method_ar": "١. طهي الكينوا بالماء حتى تصبح هشة وتركها لتبرد قليلاً.\n٢. تشويح شرائح جبنة الحلوم في مقلاة غير لاصقة حتى تكتسب لوناً ذهبياً.\n٣. تقليب الكينوا مع السبانخ، الجرجير، الخيار، وزيت الزيتون.\n٤. وضع الحلوم المشوي الدافئ على الوجه ورش القليل من السماق.",
        "ingredients": [
            {"id": 9, "base_g": 50},
            {"id": 4, "base_g": 85},
            {"id": 13, "base_g": 5},
            {"id": 17, "base_g": 40},
            {"id": 16, "base_g": 40},
            {"id": 20, "base_g": 2},
        ],
    },
    # Snack Options
    {
        "id": 401,
        "slot": "snack",
        "name_en": "Greek Yogurt & Fresh Berry Parfait",
        "name_ar": "زبادي يوناني مثلج مع التوت المشكل واللوز",
        "method_en": "1. Spoon cold Greek yogurt into a glass or bowl.\n2. Layer washed fresh berries over the top.\n3. Scatter raw almonds for crunch.",
        "method_ar": "١. وضع الزبادي اليوناني البارد في وعاء التقديم.\n٢. توزيع حبات التوت المشكل الطازج على الوجه.\n٣. إضافة حبات اللوز النيء وتقديمها مباشرة.",
        "ingredients": [
            {"id": 6, "base_g": 150},
            {"id": 18, "base_g": 70},
            {"id": 15, "base_g": 15},
        ],
    },
]

print(
    f" Loaded {len(INGREDIENTS_DB_V2)} standardized ingredients &"
    f" {len(RECIPES_DB_V2)} recipes into Munch Me mockup engine."
)

 Loaded 20 standardized ingredients & 6 recipes into Munch Me mockup engine.


In [26]:
from IPython.display import HTML, display

APP_HTML = """
<!DOCTYPE html>
<html lang="en" dir="ltr">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Munch Me | Dynamic Nutrition Engine</title>
  <script src="https://cdn.tailwindcss.com"></script>
  <link href="https://fonts.googleapis.com/css2?family=Cairo:wght@400;600;700;800&family=Inter:wght@400;500;600;700&display=swap" rel="stylesheet">
  <style>
    body { font-family: 'Inter', sans-serif; }
    [dir="rtl"] body, [dir="rtl"] { font-family: 'Cairo', sans-serif; }
    .gauge-fill { transition: width 0.4s cubic-bezier(0.4, 0, 0.2, 1); }
  </style>
</head>
<body class="bg-slate-50 text-slate-800 min-h-screen pb-16">

  <!-- TOP HEADER -->
  <header class="bg-white border-b border-slate-200 sticky top-0 z-30 px-6 py-4 shadow-sm flex items-center justify-between">
    <div class="flex items-center space-x-3 rtl:space-x-reverse">
      <div class="w-10 h-10 rounded-xl bg-emerald-600 text-white flex items-center justify-center font-black text-xl tracking-tighter">M</div>
      <div>
        <h1 class="text-xl font-bold text-slate-900 tracking-tight">Munch Me</h1>
        <p class="text-xs text-slate-400" data-i18n="tagline">Personalized Nutrition Engine</p>
      </div>
    </div>
    <!-- Language Toggle -->
    <div class="flex items-center space-x-2 rtl:space-x-reverse bg-slate-100 p-1 rounded-lg border border-slate-200 text-xs font-semibold">
      <button onclick="setLanguage('en')" id="btn-lang-en" class="px-3 py-1.5 rounded-md bg-white text-emerald-700 shadow-sm">English</button>
      <button onclick="setLanguage('ar')" id="btn-lang-ar" class="px-3 py-1.5 rounded-md text-slate-600 hover:text-slate-900">العربية</button>
    </div>
  </header>

  <main class="max-w-6xl mx-auto p-4 md:p-8 space-y-8">

    <!-- STEP 1: USER ONBOARDING -->
    <section id="onboarding-card" class="bg-white rounded-2xl p-6 md:p-8 border border-slate-200 shadow-sm space-y-6">
      <div class="border-b border-slate-100 pb-4">
        <h2 class="text-lg font-bold text-slate-900" data-i18n="profile_title">1. Your Profile & Dietary Target</h2>
        <p class="text-sm text-slate-500" data-i18n="profile_desc">Enter physiological details to compute your clinical BMR & snap your nutrition plan.</p>
      </div>

      <form id="profile-form" onsubmit="handleOnboarding(event)" class="grid grid-cols-1 md:grid-cols-3 gap-6">
        <div>
          <label class="block text-xs font-bold text-slate-600 uppercase mb-2" data-i18n="full_name">Full Name</label>
          <input type="text" id="inp-name" required value="Sarah" class="w-full px-4 py-2.5 bg-slate-50 border border-slate-200 rounded-xl focus:ring-2 focus:ring-emerald-500 text-sm outline-none">
        </div>

        <div>
          <label class="block text-xs font-bold text-slate-600 uppercase mb-2" data-i18n="dob">Date of Birth</label>
          <input type="date" id="inp-dob" required value="1996-04-15" class="w-full px-4 py-2.5 bg-slate-50 border border-slate-200 rounded-xl focus:ring-2 focus:ring-emerald-500 text-sm outline-none">
        </div>

        <div>
          <label class="block text-xs font-bold text-slate-600 uppercase mb-2" data-i18n="sex">Biological Sex</label>
          <select id="inp-sex" class="w-full px-4 py-2.5 bg-slate-50 border border-slate-200 rounded-xl focus:ring-2 focus:ring-emerald-500 text-sm outline-none">
            <option value="female" data-i18n="female">Female</option>
            <option value="male" data-i18n="male">Male</option>
          </select>
        </div>

        <div>
          <label class="block text-xs font-bold text-slate-600 uppercase mb-2" data-i18n="height">Height (cm)</label>
          <input type="number" id="inp-height" required value="165" class="w-full px-4 py-2.5 bg-slate-50 border border-slate-200 rounded-xl focus:ring-2 focus:ring-emerald-500 text-sm outline-none">
        </div>

        <div>
          <label class="block text-xs font-bold text-slate-600 uppercase mb-2" data-i18n="weight">Weight (kg)</label>
          <input type="number" step="0.5" id="inp-weight" required value="68" class="w-full px-4 py-2.5 bg-slate-50 border border-slate-200 rounded-xl focus:ring-2 focus:ring-emerald-500 text-sm outline-none">
        </div>

        <div>
          <label class="block text-xs font-bold text-slate-600 uppercase mb-2" data-i18n="activity">Physical Activity Level (PAL)</label>
          <select id="inp-pal" class="w-full px-4 py-2.5 bg-slate-50 border border-slate-200 rounded-xl focus:ring-2 focus:ring-emerald-500 text-sm outline-none">
            <option value="1.2" data-i18n="pal_1">Sedentary (1.2) - Desk Job / No Exercise</option>
            <option value="1.375" selected data-i18n="pal_2">Lightly Active (1.375) - 1-3 days/wk</option>
            <option value="1.55" data-i18n="pal_3">Moderately Active (1.55) - 3-5 days/wk</option>
            <option value="1.725" data-i18n="pal_4">Very Active (1.725) - 6-7 days/wk</option>
          </select>
        </div>

        <div>
          <label class="block text-xs font-bold text-slate-600 uppercase mb-2" data-i18n="primary_goal">Primary Goal</label>
          <select id="inp-goal" class="w-full px-4 py-2.5 bg-slate-50 border border-slate-200 rounded-xl focus:ring-2 focus:ring-emerald-500 text-sm outline-none">
            <option value="weight_loss" data-i18n="goal_loss">Weight Loss (45% C / 30% P / 25% F)</option>
            <option value="high_protein" data-i18n="goal_protein">High Protein (40% C / 35% P / 25% F)</option>
            <option value="standard" data-i18n="goal_standard">Standard Balance (50% C / 25% P / 25% F)</option>
          </select>
        </div>

        <div>
          <label class="block text-xs font-bold text-slate-600 uppercase mb-2" data-i18n="country">Country (Pricing & Currency)</label>
          <select id="inp-country" class="w-full px-4 py-2.5 bg-slate-50 border border-slate-200 rounded-xl focus:ring-2 focus:ring-emerald-500 text-sm outline-none">
            <option value="JO" selected data-i18n="country_jo">Jordan (JOD د.أ)</option>
            <option value="SA" data-i18n="country_sa">Saudi Arabia (SAR ر.س)</option>
            <option value="AE" data-i18n="country_ae">United Arab Emirates (AED د.إ)</option>
          </select>
        </div>

        <div>
          <label class="block text-xs font-bold text-slate-600 uppercase mb-2" data-i18n="split_structure">Daily Meal Routine</label>
          <select id="inp-split" class="w-full px-4 py-2.5 bg-slate-50 border border-slate-200 rounded-xl focus:ring-2 focus:ring-emerald-500 text-sm outline-none">
            <option value="b_l_d" data-i18n="routine_1">Breakfast, Lunch, Dinner</option>
            <option value="l_d_s" data-i18n="routine_2">Lunch, Dinner, Snack</option>
            <option value="b_l_s" data-i18n="routine_3">Breakfast, Lunch, Snack</option>
            <option value="b_l_d_s" selected data-i18n="routine_4">Breakfast, Lunch, Dinner, 1 Snack</option>
            <option value="full_5" data-i18n="routine_5">Breakfast, Lunch, Dinner, 2 Snacks</option>
          </select>
        </div>

        <div class="md:col-span-3 pt-2">
          <button type="submit" class="w-full py-3.5 bg-emerald-600 hover:bg-emerald-700 text-white font-bold rounded-xl shadow-sm transition-all" data-i18n="calculate_btn">
            Compute Caloric Target & Build Plan
          </button>
        </div>
      </form>
    </section>

    <!-- STEP 2: DEPLETION METERS -->
    <section id="meters-card" class="hidden bg-white rounded-2xl p-6 md:p-8 border border-slate-200 shadow-sm space-y-6">
      <div class="flex flex-col md:flex-row md:items-center justify-between border-b border-slate-100 pb-4 gap-2">
        <div>
          <h2 class="text-lg font-bold text-slate-900" data-i18n="meters_title">Daily Balance & Remaining Allowance</h2>
          <p class="text-sm text-slate-500" data-i18n="meters_desc">Real-time depletion meters that update automatically as you lock in meals.</p>
        </div>
        <div id="status-badge" class="px-3 py-1 bg-emerald-50 border border-emerald-200 rounded-lg text-xs font-bold text-emerald-800 self-start md:self-auto"></div>
      </div>

      <!-- Gauges Grid -->
      <div class="grid grid-cols-1 sm:grid-cols-2 lg:grid-cols-5 gap-4">
        <!-- Calories -->
        <div class="p-4 rounded-xl bg-slate-50 border border-slate-200">
          <div class="flex justify-between text-xs font-semibold text-slate-500 mb-1">
            <span data-i18n="calories">Calories</span>
            <span id="cal-remain-label">0 kcal left</span>
          </div>
          <div class="text-xl font-black text-slate-900 mb-2"><span id="cal-logged">0</span> / <span id="cal-target">0</span> <span class="text-xs font-normal">kcal</span></div>
          <div class="w-full bg-slate-200 h-2 rounded-full overflow-hidden">
            <div id="cal-bar" class="gauge-fill bg-emerald-500 h-full w-0"></div>
          </div>
        </div>

        <!-- Protein -->
        <div class="p-4 rounded-xl bg-slate-50 border border-slate-200">
          <div class="flex justify-between text-xs font-semibold text-slate-500 mb-1">
            <span data-i18n="protein">Protein</span>
            <span id="pro-remain-label">0g left</span>
          </div>
          <div class="text-xl font-black text-slate-900 mb-2"><span id="pro-logged">0</span> / <span id="pro-target">0</span> <span class="text-xs font-normal">g</span></div>
          <div class="w-full bg-slate-200 h-2 rounded-full overflow-hidden">
            <div id="pro-bar" class="gauge-fill bg-blue-500 h-full w-0"></div>
          </div>
        </div>

        <!-- Carbs -->
        <div class="p-4 rounded-xl bg-slate-50 border border-slate-200">
          <div class="flex justify-between text-xs font-semibold text-slate-500 mb-1">
            <span data-i18n="carbs">Carbs</span>
            <span id="carb-remain-label">0g left</span>
          </div>
          <div class="text-xl font-black text-slate-900 mb-2"><span id="carb-logged">0</span> / <span id="carb-target">0</span> <span class="text-xs font-normal">g</span></div>
          <div class="w-full bg-slate-200 h-2 rounded-full overflow-hidden">
            <div id="carb-bar" class="gauge-fill bg-amber-500 h-full w-0"></div>
          </div>
        </div>

        <!-- Fat -->
        <div class="p-4 rounded-xl bg-slate-50 border border-slate-200">
          <div class="flex justify-between text-xs font-semibold text-slate-500 mb-1">
            <span data-i18n="fat">Fat</span>
            <span id="fat-remain-label">0g left</span>
          </div>
          <div class="text-xl font-black text-slate-900 mb-2"><span id="fat-logged">0</span> / <span id="fat-target">0</span> <span class="text-xs font-normal">g</span></div>
          <div class="w-full bg-slate-200 h-2 rounded-full overflow-hidden">
            <div id="fat-bar" class="gauge-fill bg-rose-500 h-full w-0"></div>
          </div>
        </div>

        <!-- Water Tracker -->
        <div class="p-4 rounded-xl bg-cyan-50/60 border border-cyan-200">
          <div class="flex justify-between text-xs font-semibold text-cyan-800 mb-1">
            <span data-i18n="water">Water Tracker</span>
            <span id="water-remain-label">0 ml left</span>
          </div>
          <div class="text-xl font-black text-cyan-950 mb-2"><span id="water-logged">0</span> / <span id="water-target">0</span> <span class="text-xs font-normal">ml</span></div>
          <div class="flex items-center space-x-2 rtl:space-x-reverse">
            <div class="w-full bg-cyan-200 h-2 rounded-full overflow-hidden">
              <div id="water-bar" class="gauge-fill bg-cyan-600 h-full w-0"></div>
            </div>
            <button onclick="logWater(250)" class="text-xs font-bold bg-cyan-600 hover:bg-cyan-700 text-white px-2 py-0.5 rounded shadow-sm">+250</button>
          </div>
        </div>
      </div>
    </section>

    <!-- STEP 3: DYNAMIC MEAL SLOTS BUILDER -->
    <section id="slots-section" class="hidden space-y-6">
      <div class="flex items-center justify-between">
        <h2 class="text-lg font-bold text-slate-900" data-i18n="meal_schedule">Daily Meal Schedule & 1g Scaled Recipes</h2>
        <span class="text-xs text-slate-500" data-i18n="scaled_desc">Quantities computed in exact integer grams</span>
      </div>
      <div id="slots-container" class="space-y-6"></div>
    </section>

    <!-- STEP 4: AGGREGATED GROCERY LIST SORTED BY AISLE -->
    <section id="grocery-section" class="hidden bg-white rounded-2xl p-6 md:p-8 border border-slate-200 shadow-sm space-y-6">
      <div class="flex flex-col md:flex-row md:items-center justify-between border-b border-slate-100 pb-4 gap-2">
        <div>
          <h2 class="text-lg font-bold text-slate-900" data-i18n="grocery_title">Consolidated Grocery List & Spend</h2>
          <p class="text-sm text-slate-500" data-i18n="grocery_desc">Aggregated raw ingredients calculated directly from your scheduled portions, organized by supermarket department.</p>
        </div>
        <div class="text-right rtl:text-left">
          <span class="text-xs text-slate-500" data-i18n="total_estimated_cost">Total Estimated Cost</span>
          <div class="text-2xl font-black text-emerald-600" id="grocery-total-cost">0.00 JOD</div>
        </div>
      </div>

      <div id="grocery-departments-container" class="space-y-6"></div>
    </section>

  </main>

  <script>
    // ==========================================
    // 1. DATA MATRICES & MOCKUP DATA
    // ==========================================
    const PLAN_MATRIX = [
      // Weight Loss (45% C, 30% P, 25% F)
      { plan_type: "weight_loss", calories: 800, carbs_g: 90.0, protein_g: 60.0, fat_g: 22.2 },
      { plan_type: "weight_loss", calories: 1000, carbs_g: 112.5, protein_g: 75.0, fat_g: 27.8 },
      { plan_type: "weight_loss", calories: 1200, carbs_g: 135.0, protein_g: 90.0, fat_g: 33.3 },
      { plan_type: "weight_loss", calories: 1400, carbs_g: 157.5, protein_g: 105.0, fat_g: 38.9 },
      { plan_type: "weight_loss", calories: 1600, carbs_g: 180.0, protein_g: 120.0, fat_g: 44.4 },
      { plan_type: "weight_loss", calories: 1800, carbs_g: 202.5, protein_g: 135.0, fat_g: 50.0 },
      { plan_type: "weight_loss", calories: 2000, carbs_g: 225.0, protein_g: 150.0, fat_g: 55.6 },
      // High Protein (40% C, 35% P, 25% F)
      { plan_type: "high_protein", calories: 1600, carbs_g: 160.0, protein_g: 140.0, fat_g: 44.4 },
      { plan_type: "high_protein", calories: 1800, carbs_g: 180.0, protein_g: 157.5, fat_g: 50.0 },
      { plan_type: "high_protein", calories: 2000, carbs_g: 200.0, protein_g: 175.0, fat_g: 55.6 },
      { plan_type: "high_protein", calories: 2500, carbs_g: 250.0, protein_g: 218.75, fat_g: 69.4 },
      { plan_type: "high_protein", calories: 3000, carbs_g: 300.0, protein_g: 262.5, fat_g: 83.3 },
      // Standard (50% C, 25% P, 25% F)
      { plan_type: "standard", calories: 1400, carbs_g: 175.0, protein_g: 87.5, fat_g: 38.9 },
      { plan_type: "standard", calories: 1600, carbs_g: 200.0, protein_g: 100.0, fat_g: 44.4 },
      { plan_type: "standard", calories: 1800, carbs_g: 225.0, protein_g: 112.5, fat_g: 50.0 },
      { plan_type: "standard", calories: 2000, carbs_g: 250.0, protein_g: 125.0, fat_g: 55.6 },
      { plan_type: "standard", calories: 2500, carbs_g: 312.5, protein_g: 156.25, fat_g: 69.4 }
    ];

    const INGREDIENTS_DB = {
      1: { id: 1, name_en: "Chicken Breast (Raw)", name_ar: "صدر دجاج (نيء)", dept_en: "Meat & Poultry", dept_ar: "لحوم ودواجن", cal_per_g: 1.20, pro_per_g: 0.225, carb_per_g: 0.0, fat_per_g: 0.026 },
      2: { id: 2, name_en: "Lean Ground Beef (90/10)", name_ar: "لحم بقر مفروم قليل الدهن", dept_en: "Meat & Poultry", dept_ar: "لحوم ودواجن", cal_per_g: 1.76, pro_per_g: 0.200, carb_per_g: 0.0, fat_per_g: 0.100 },
      3: { id: 3, name_en: "Whole Eggs", name_ar: "بيض طازج", dept_en: "Dairy & Eggs", dept_ar: "ألبان وبيض", cal_per_g: 1.43, pro_per_g: 0.126, carb_per_g: 0.008, fat_per_g: 0.095 },
      4: { id: 4, name_en: "Low-Fat Halloumi Cheese", name_ar: "جبنة حلوم قليلة الدسم", dept_en: "Dairy & Eggs", dept_ar: "ألبان وبيض", cal_per_g: 2.60, pro_per_g: 0.220, carb_per_g: 0.020, fat_per_g: 0.180 },
      5: { id: 5, name_en: "Traditional Labneh (Low-Fat)", name_ar: "لبنة بلدية قليلة الدسم", dept_en: "Dairy & Eggs", dept_ar: "ألبان وبيض", cal_per_g: 1.05, pro_per_g: 0.090, carb_per_g: 0.040, fat_per_g: 0.055 },
      6: { id: 6, name_en: "Greek Yogurt (0% Fat)", name_ar: "لبن زبادي يوناني (خالي الدسم)", dept_en: "Dairy & Eggs", dept_ar: "ألبان وبيض", cal_per_g: 0.59, pro_per_g: 0.100, carb_per_g: 0.036, fat_per_g: 0.004 },
      7: { id: 7, name_en: "Basmati White Rice (Raw)", name_ar: "أرز بسمتي أبيض (غير مطبوخ)", dept_en: "Pantry & Grains", dept_ar: "حبوب وبقوليات", cal_per_g: 3.60, pro_per_g: 0.070, carb_per_g: 0.800, fat_per_g: 0.007 },
      8: { id: 8, name_en: "Whole Green Freekeh (Dry)", name_ar: "فريكة خضراء حب (جافة)", dept_en: "Pantry & Grains", dept_ar: "حبوب وبقوليات", cal_per_g: 3.25, pro_per_g: 0.145, carb_per_g: 0.650, fat_per_g: 0.022 },
      9: { id: 9, name_en: "Quinoa (Dry)", name_ar: "كينوا (جافة)", dept_en: "Pantry & Grains", dept_ar: "حبوب وبقوليات", cal_per_g: 3.68, pro_per_g: 0.141, carb_per_g: 0.642, fat_per_g: 0.061 },
      10: { id: 10, name_en: "Rolled Oats", name_ar: "شوفان حبة كاملة", dept_en: "Pantry & Grains", dept_ar: "حبوب وبقوليات", cal_per_g: 3.89, pro_per_g: 0.169, carb_per_g: 0.663, fat_per_g: 0.069 },
      11: { id: 11, name_en: "Whole Wheat Sourdough Bread", name_ar: "خبز قمح كامل مخمر", dept_en: "Bakery", dept_ar: "مخبوزات", cal_per_g: 2.38, pro_per_g: 0.110, carb_per_g: 0.440, fat_per_g: 0.020 },
      12: { id: 12, name_en: "Sweet Potato (Raw)", name_ar: "بطاطا حلوة نيئة", dept_en: "Produce", dept_ar: "خضار وفواكه", cal_per_g: 0.86, pro_per_g: 0.016, carb_per_g: 0.201, fat_per_g: 0.001 },
      13: { id: 13, name_en: "Extra Virgin Olive Oil", name_ar: "زيت زيتون بكر ممتاز", dept_en: "Pantry & Grains", dept_ar: "حبوب وبقوليات", cal_per_g: 8.84, pro_per_g: 0.0, carb_per_g: 0.0, fat_per_g: 1.000 },
      14: { id: 14, name_en: "Tahini Paste (100% Sesame)", name_ar: "طحينية سمسم نقية", dept_en: "Pantry & Grains", dept_ar: "حبوب وبقوليات", cal_per_g: 5.95, pro_per_g: 0.170, carb_per_g: 0.210, fat_per_g: 0.540 },
      15: { id: 15, name_en: "Raw Almonds", name_ar: "لوز نيء", dept_en: "Pantry & Grains", dept_ar: "حبوب وبقوليات", cal_per_g: 5.79, pro_per_g: 0.212, carb_per_g: 0.216, fat_per_g: 0.499 },
      16: { id: 16, name_en: "Fresh Cucumber & Mint", name_ar: "خيار ونعناع طازج", dept_en: "Produce", dept_ar: "خضار وفواكه", cal_per_g: 0.15, pro_per_g: 0.007, carb_per_g: 0.036, fat_per_g: 0.001 },
      17: { id: 17, name_en: "Baby Spinach & Arugula", name_ar: "سبانخ وجرجير طازج", dept_en: "Produce", dept_ar: "خضار وفواكه", cal_per_g: 0.23, pro_per_g: 0.029, carb_per_g: 0.036, fat_per_g: 0.004 },
      18: { id: 18, name_en: "Fresh Mixed Berries", name_ar: "توت مشكل طازج", dept_en: "Produce", dept_ar: "خضار وفواكه", cal_per_g: 0.57, pro_per_g: 0.007, carb_per_g: 0.140, fat_per_g: 0.003 },
      19: { id: 19, name_en: "Shawarma Spice Blend & Garlic", name_ar: "بهارات شاورما وثوم مهروس", dept_en: "Spices & Seasoning", dept_ar: "بهارات وتوابل", cal_per_g: 0.0, pro_per_g: 0.0, carb_per_g: 0.0, fat_per_g: 0.0 },
      20: { id: 20, name_en: "Za'atar & Sumac Blend", name_ar: "خلطة زعتر بلدي وسماق", dept_en: "Spices & Seasoning", dept_ar: "بهارات وتوابل", cal_per_g: 0.0, pro_per_g: 0.0, carb_per_g: 0.0, fat_per_g: 0.0 }
    };

    const REGIONAL_PRICING = {
      JO: { currency: "JOD", 1: 0.0045, 2: 0.0075, 3: 0.0035, 4: 0.0080, 5: 0.0040, 6: 0.0050, 7: 0.0015, 8: 0.0022, 9: 0.0040, 10: 0.0030, 11: 0.0025, 12: 0.0012, 13: 0.0090, 14: 0.0055, 15: 0.0120, 16: 0.0008, 17: 0.0010, 18: 0.0080, 19: 0.0020, 20: 0.0020 },
      SA: { currency: "SAR", 1: 0.0280, 2: 0.0420, 3: 0.0180, 4: 0.0480, 5: 0.0240, 6: 0.0220, 7: 0.0090, 8: 0.0140, 9: 0.0240, 10: 0.0150, 11: 0.0120, 12: 0.0065, 13: 0.0450, 14: 0.0280, 15: 0.0650, 16: 0.0045, 17: 0.0055, 18: 0.0400, 19: 0.0080, 20: 0.0080 },
      AE: { currency: "AED", 1: 0.0270, 2: 0.0400, 3: 0.0175, 4: 0.0460, 5: 0.0230, 6: 0.0210, 7: 0.0085, 8: 0.0135, 9: 0.0230, 10: 0.0140, 11: 0.0115, 12: 0.0060, 13: 0.0440, 14: 0.0270, 15: 0.0620, 16: 0.0040, 17: 0.0050, 18: 0.0380, 19: 0.0075, 20: 0.0075 }
    };

    const RECIPES_DB = [
      {
        id: 101,
        slot: "breakfast",
        name_en: "Za'atar Labneh & Whole Wheat Sourdough",
        name_ar: "توست اللبنة البلدية بالزعتر وزيت الزيتون",
        method_en: "1. Lightly toast the sourdough bread until crisp.\\n2. Spread chilled labneh evenly over the surface.\\n3. Drizzle olive oil and dust with za'atar sumac blend.\\n4. Serve fresh with cucumber and mint.",
        method_ar: "١. تحميص خبز القمح الكامل حتى يصبح مقرمشاً.\\n٢. فرد اللبنة البلدية الباردة بالتساوي على الوجه.\\n٣. سكب زيت الزيتون ورش خلطة الزعتر والسماق.\\n٤. يُقدم مباشرة مع الخيار والنعناع الطازج.",
        ingredients: [
          { id: 11, base_g: 60 },
          { id: 5, base_g: 90 },
          { id: 13, base_g: 5 },
          { id: 20, base_g: 3 },
          { id: 16, base_g: 50 }
        ]
      },
      {
        id: 102,
        slot: "breakfast",
        name_en: "Golden Sweet Potato & Egg Skillet",
        name_ar: "مقلاة البطاطا الحلوة والبيض المشوي",
        method_en: "1. Dice sweet potato and sauté in olive oil until soft and caramelized.\\n2. Whisk whole eggs and pour over the potatoes.\\n3. Cook gently over low heat until eggs are softly set.\\n4. Serve warm.",
        method_ar: "١. تقطيع البطاطا الحلوة وتشويحها بزيت الزيتون حتى تلين وتتحمر.\\n٢. خفق البيض الطازج وسكبه فوق البطاطا.\\n٣. الطهي على نار هادئة حتى ينضج البيض تماماً.\\n٤. يُقدم الطبق ساخناً.",
        ingredients: [
          { id: 12, base_g: 120 },
          { id: 3, base_g: 110 },
          { id: 13, base_g: 6 }
        ]
      },
      {
        id: 201,
        slot: "lunch",
        name_en: "Healthy Chicken Shawarma Plate with Basmati",
        name_ar: "طبق شاورما الدجاج الصحية مع أرز بسمتي",
        method_en: "1. Rub raw chicken strips with shawarma spices.\\n2. Sear chicken in hot olive oil until thoroughly cooked and slightly charred.\\n3. Steam basmati rice.\\n4. Whisk tahini with lemon juice and a splash of water for the dressing.\\n5. Assemble plate with warm rice, spiced chicken, fresh greens, and tahini drizzle.",
        method_ar: "١. تتبيل شرائح الدجاج بخلطة بهارات الشاورما والثوم.\\n٢. تشويح الدجاج في مقلاة ساخنة بزيت الزيتون حتى يكتسب لون التحمير وينضج.\\n٣. طهي أرز البسمتي على البخار.\\n٤. خفق الطحينية مع قليل من الماء وعصير الليمون.\\n٥. ترتيب الأرز، الدجاج المحمر، الورقيات، وسكب صلصة الطحينية على الوجه.",
        ingredients: [
          { id: 1, base_g: 160 },
          { id: 7, base_g: 60 },
          { id: 13, base_g: 6 },
          { id: 14, base_g: 12 },
          { id: 19, base_g: 4 },
          { id: 17, base_g: 30 }
        ]
      },
      {
        id: 202,
        slot: "lunch",
        name_en: "Spiced Green Freekeh with Lean Beef",
        name_ar: "فريكة خضراء باللحم البقري المفروم والمكسرات",
        method_en: "1. Rinse freekeh thoroughly; simmer in water or broth until tender and smoky.\\n2. Brown lean ground beef in a skillet with olive oil and spices.\\n3. Layer cooked freekeh into a serving bowl, top with seasoned beef.\\n4. Garnish with almonds.",
        method_ar: "١. غسل الفريكة الخضراء وطهيها بالماء المغلي حتى تنضج وتمتص النكهة المدخنة.\\n٢. طهي اللحم البقري المفروم بزيت الزيتون والبهارات حتى ينضج تماماً.\\n٣. سكب الفريكة في طبق التقديم ووضع اللحم المفروم فوقها.\\n٤. التزيين بحبات اللوز وتقديمها ساخنة.",
        ingredients: [
          { id: 8, base_g: 65 },
          { id: 2, base_g: 130 },
          { id: 13, base_g: 6 },
          { id: 15, base_g: 12 },
          { id: 19, base_g: 3 }
        ]
      },
      {
        id: 301,
        slot: "dinner",
        name_en: "Grilled Halloumi & Herb Quinoa Bowl",
        name_ar: "وعاء الكينوا بجبنة الحلوم المشوية والنعناع",
        method_en: "1. Simmer quinoa in water until fluffy, then allow to cool slightly.\\n2. Sear halloumi cheese slices in a dry non-stick skillet until golden crust forms.\\n3. Toss quinoa with fresh spinach, arugula, diced cucumber, and olive oil.\\n4. Top with warm grilled halloumi and dust with sumac.",
        method_ar: "١. طهي الكينوا بالماء حتى تصبح هشة وتركها لتبرد قليلاً.\\n٢. تشويح شرائح جبنة الحلوم في مقلاة غير لاصقة حتى تكتسب لوناً ذهبياً.\\n٣. تقليب الكينوا مع السبانخ، الجرجير، الخيار، وزيت الزيتون.\\n٤. وضع الحلوم المشوي الدافئ على الوجه ورش القليل من السماق.",
        ingredients: [
          { id: 9, base_g: 50 },
          { id: 4, base_g: 85 },
          { id: 13, base_g: 5 },
          { id: 17, base_g: 40 },
          { id: 16, base_g: 40 },
          { id: 20, base_g: 2 }
        ]
      },
      {
        id: 401,
        slot: "snack",
        name_en: "Greek Yogurt & Fresh Berry Parfait",
        name_ar: "زبادي يوناني مثلج مع التوت المشكل واللوز",
        method_en: "1. Spoon cold Greek yogurt into a glass or bowl.\\n2. Layer washed fresh berries over the top.\\n3. Scatter raw almonds for crunch.",
        method_ar: "١. وضع الزبادي اليوناني البارد في وعاء التقديم.\\n٢. توزيع حبات التوت المشكل الطازج على الوجه.\\n٣. إضافة حبات اللوز النيء وتقديمها مباشرة.",
        ingredients: [
          { id: 6, base_g: 150 },
          { id: 18, base_g: 70 },
          { id: 15, base_g: 15 }
        ]
      }
    ];

    // Pre-calculate baseline totals
    RECIPES_DB.forEach(r => {
      let cal = 0, pro = 0, carb = 0, fat = 0;
      r.ingredients.forEach(item => {
        const ref = INGREDIENTS_DB[item.id];
        cal += item.base_g * ref.cal_per_g;
        pro += item.base_g * ref.pro_per_g;
        carb += item.base_g * ref.carb_per_g;
        fat += item.base_g * ref.fat_per_g;
      });
      r.base_cal = cal;
      r.base_pro = pro;
      r.base_carb = carb;
      r.base_fat = fat;
    });

    const ROUTINE_SPLITS = {
      b_l_d: { breakfast: 0.30, lunch: 0.40, dinner: 0.30 },
      l_d_s: { lunch: 0.425, dinner: 0.425, snack_1: 0.15 },
      b_l_s: { breakfast: 0.425, lunch: 0.425, snack_1: 0.15 },
      b_l_d_s: { breakfast: 0.25, lunch: 0.35, dinner: 0.25, snack_1: 0.15 },
      full_5: { breakfast: 0.25, lunch: 0.35, dinner: 0.25, snack_1: 0.075, snack_2: 0.075 }
    };

    // ==========================================
    // 2. TRANSLATION STRINGS
    // ==========================================
    let currentLang = 'en';
    const translations = {
      en: {
        tagline: "Personalized Nutrition Engine",
        profile_title: "1. Your Profile & Dietary Target",
        profile_desc: "Enter physiological details to compute your clinical BMR & snap your nutrition plan.",
        full_name: "Full Name",
        dob: "Date of Birth",
        sex: "Biological Sex",
        female: "Female",
        male: "Male",
        height: "Height (cm)",
        weight: "Weight (kg)",
        activity: "Physical Activity Level (PAL)",
        pal_1: "Sedentary (1.2) - Desk Job / No Exercise",
        pal_2: "Lightly Active (1.375) - 1-3 days/wk",
        pal_3: "Moderately Active (1.55) - 3-5 days/wk",
        pal_4: "Very Active (1.725) - 6-7 days/wk",
        primary_goal: "Primary Goal",
        goal_loss: "Weight Loss (45% C / 30% P / 25% F)",
        goal_protein: "High Protein (40% C / 35% P / 25% F)",
        goal_standard: "Standard Balance (50% C / 25% P / 25% F)",
        country: "Country (Pricing & Currency)",
        country_jo: "Jordan (JOD د.أ)",
        country_sa: "Saudi Arabia (SAR ر.س)",
        country_ae: "United Arab Emirates (AED د.إ)",
        split_structure: "Daily Meal Routine",
        routine_1: "Breakfast, Lunch, Dinner",
        routine_2: "Lunch, Dinner, Snack",
        routine_3: "Breakfast, Lunch, Snack",
        routine_4: "Breakfast, Lunch, Dinner, 1 Snack",
        routine_5: "Breakfast, Lunch, Dinner, 2 Snacks",
        calculate_btn: "Compute Caloric Target & Build Plan",
        meters_title: "Daily Balance & Remaining Allowance",
        meters_desc: "Real-time depletion meters that update automatically as you lock in meals.",
        calories: "Calories",
        protein: "Protein",
        carbs: "Carbohydrates",
        fat: "Fat",
        water: "Water Tracker",
        left: "left",
        over: "over target",
        meal_schedule: "Daily Meal Schedule & 1g Scaled Recipes",
        scaled_desc: "Quantities computed in exact integer grams",
        grocery_title: "Consolidated Grocery List & Spend",
        grocery_desc: "Aggregated raw ingredients calculated directly from your scheduled portions, organized by supermarket department.",
        total_estimated_cost: "Total Estimated Cost",
        pantry_status: "Pantry",
        ingredient: "Ingredient",
        scaled_weight: "Total Weight",
        cost: "Regional Cost",
        select_recipe: "Select This Meal",
        selected_badge: "Selected",
        method_title: "Preparation Method:",
        breakfast: "Breakfast",
        lunch: "Lunch",
        dinner: "Dinner",
        snack_1: "Snack 1",
        snack_2: "Snack 2"
      },
      ar: {
        tagline: "منظومة التغذية الذكية وتخطيط الوجبات",
        profile_title: "١. الملف الشخصي والأهداف الغذائية",
        profile_desc: "أدخل بياناتك الفسيولوجية لحساب معدل الأيض الأساسي (BMR) وتحديد خطتك بدقة.",
        full_name: "الاسم الكامل",
        dob: "تاريخ الميلاد",
        sex: "الجنس",
        female: "أنثى",
        male: "ذكر",
        height: "الطول (سم)",
        weight: "الوزن (كغ)",
        activity: "مستوى النشاط البدني",
        pal_1: "خامل (١.٢) - عمل مكتبي / بدون تمارين",
        pal_2: "نشاط خفيف (١.٣٧٥) - تمرين ١-٣ أيام/أسبوع",
        pal_3: "نشاط متوسط (١.٥٥) - تمرين ٣-٥ أيام/أسبوع",
        pal_4: "نشاط عالي (١.٧٢٥) - تمرين ٦-٧ أيام/أسبوع",
        primary_goal: "الهدف الرئيسي",
        goal_loss: "خسارة وزن (٤٥٪ كربوهيدرات / ٣٠٪ بروتين / ٢٥٪ دهون)",
        goal_protein: "عالي البروتين (٤٠٪ كربوهيدرات / ٣٥٪ بروتين / ٢٥٪ دهون)",
        goal_standard: "متوازن قياسي (٥٠٪ كربوهيدرات / ٢٥٪ بروتين / ٢٥٪ دهون)",
        country: "البلد (التسعير والعملة)",
        country_jo: "الأردن (JOD د.أ)",
        country_sa: "المملكة العربية السعودية (SAR ر.س)",
        country_ae: "الإمارات العربية المتحدة (AED د.إ)",
        split_structure: "روتين الوجبات اليومي",
        routine_1: "فطور، غداء، عشاء",
        routine_2: "غداء، عشاء، وجبة خفيفة",
        routine_3: "فطور، غداء، وجبة خفيفة",
        routine_4: "فطور، غداء، عشاء، وجبة خفيفة ١",
        routine_5: "فطور، غداء، عشاء، وجبتان خفيفتان",
        calculate_btn: "احتساب السعرات وتوليد الخطة",
        meters_title: "مؤشرات الاستهلاك والمتبقي اليومي",
        meters_desc: "عدادات تتبع تفاعلية تُحدّث تلقائياً لمعرفة المتبقي من حصتك الغذائية.",
        calories: "السعرات",
        protein: "البروتين",
        carbs: "الكربوهيدرات",
        fat: "الدهون",
        water: "متابعة شرب الماء",
        left: "متبقٍ",
        over: "فوق الهدف",
        meal_schedule: "جدول الوجبات اليومية والمقادير الموزونة",
        scaled_desc: "أوزان ومقادير محسوبة بدقة الغرام الواحد",
        grocery_title: "قائمة المشتريات المجمعة والتكلفة",
        grocery_desc: "تجميع ذكي لجميع المكونات النيئة المحسوبة عبر وجباتك المقررة، مقسمة حسب أقسام السوبرماركت.",
        total_estimated_cost: "التكلفة الإجمالية التقديرية",
        pantry_status: "متوفر في المنزل",
        ingredient: "المكون",
        scaled_weight: "الوزن الكلي",
        cost: "التكلفة",
        select_recipe: "اختيار هذه الوجبة",
        selected_badge: "تم الاختيار",
        method_title: "طريقة التحضير:",
        breakfast: "الفطور",
        lunch: "الغداء",
        dinner: "العشاء",
        snack_1: "وجبة خفيفة ١",
        snack_2: "وجبة خفيفة ٢"
      }
    };

    function setLanguage(lang) {
      currentLang = lang;
      document.documentElement.lang = lang;
      document.documentElement.dir = lang === 'ar' ? 'rtl' : 'ltr';

      document.getElementById('btn-lang-en').className = lang === 'en'
        ? 'px-3 py-1.5 rounded-md bg-white text-emerald-700 shadow-sm'
        : 'px-3 py-1.5 rounded-md text-slate-600 hover:text-slate-900';
      document.getElementById('btn-lang-ar').className = lang === 'ar'
        ? 'px-3 py-1.5 rounded-md bg-white text-emerald-700 shadow-sm'
        : 'px-3 py-1.5 rounded-md text-slate-600 hover:text-slate-900';

      document.querySelectorAll('[data-i18n]').forEach(elem => {
        const key = elem.getAttribute('data-i18n');
        if (translations[lang][key]) {
          elem.innerText = translations[lang][key];
        }
      });

      if (appState.plan) {
        renderMeters();
        renderAllSlots();
        renderGroceryList();
      }
    }

    // ==========================================
    // 3. APPLICATION STATE & SCALING ENGINE
    // ==========================================
    let appState = {
      user: null,
      plan: null,
      routineKey: 'b_l_d_s',
      selectedMeals: {}, // slot -> scaled recipe object
      loggedWater: 0
    };

    function handleOnboarding(e) {
      e.preventDefault();
      const dobVal = document.getElementById('inp-dob').value;
      const dob = new Date(dobVal);
      const today = new Date();
      let age = today.getFullYear() - dob.getFullYear();
      const m = today.getMonth() - dob.getMonth();
      if (m < 0 || (m === 0 && today.getDate() < dob.getDate())) {
        age--;
      }

      const weight = parseFloat(document.getElementById('inp-weight').value);
      const height = parseFloat(document.getElementById('inp-height').value);
      const sex = document.getElementById('inp-sex').value;
      const pal = parseFloat(document.getElementById('inp-pal').value);
      const goal = document.getElementById('inp-goal').value;
      const country = document.getElementById('inp-country').value;
      const routineKey = document.getElementById('inp-split').value;

      // Mifflin-St Jeor Formula
      let bmr = (10 * weight) + (6.25 * height) - (5 * age);
      bmr += (sex === 'male') ? 5 : -161;
      const tdee = bmr * pal;

      let targetKcal = (goal === 'weight_loss') ? tdee - 500 : (goal === 'high_protein') ? tdee + 300 : tdee;

      // Snap to nearest row in matrix
      const candidates = PLAN_MATRIX.filter(p => p.plan_type === goal);
      let snapped = candidates[0];
      let minDiff = Math.abs(candidates[0].calories - targetKcal);
      candidates.forEach(c => {
        const diff = Math.abs(c.calories - targetKcal);
        if (diff < minDiff) {
          minDiff = diff;
          snapped = c;
        }
      });

      appState.user = { weight, height, sex, pal, goal, country, age };
      appState.routineKey = routineKey;
      appState.plan = {
        bmr: Math.round(bmr),
        tdee: Math.round(tdee),
        target_calories: snapped.calories,
        target_carbs: snapped.carbs_g,
        target_protein: snapped.protein_g,
        target_fat: snapped.fat_g,
        water_target: Math.round(weight * 35)
      };
      appState.selectedMeals = {};
      appState.loggedWater = 0;

      document.getElementById('meters-card').classList.remove('hidden');
      document.getElementById('slots-section').classList.remove('hidden');
      document.getElementById('grocery-section').classList.remove('hidden');

      buildInitialMeals();
    }

    // Dynamic 1g scale calculation
    function scaleRecipe(recipe, targetKcal, country) {
      const scale = targetKcal / recipe.base_cal;
      const currency = REGIONAL_PRICING[country].currency;
      const costs = REGIONAL_PRICING[country];

      let mealCost = 0;
      const scaledIngredients = recipe.ingredients.map(item => {
        const ref = INGREDIENTS_DB[item.id];
        const scaledWeight = Math.round(item.base_g * scale);
        const cost = scaledWeight * (costs[item.id] || 0);
        mealCost += cost;

        return {
          id: item.id,
          name_en: ref.name_en,
          name_ar: ref.name_ar,
          dept_en: ref.dept_en,
          dept_ar: ref.dept_ar,
          weight_g: scaledWeight,
          cost: cost
        };
      });

      return {
        id: recipe.id,
        slot: recipe.slot,
        name_en: recipe.name_en,
        name_ar: recipe.name_ar,
        method_en: recipe.method_en,
        method_ar: recipe.method_ar,
        calories: Math.round(recipe.base_cal * scale),
        protein: Math.round(recipe.base_pro * scale * 10) / 10,
        carbs: Math.round(recipe.base_carb * scale * 10) / 10,
        fat: Math.round(recipe.base_fat * scale * 10) / 10,
        cost: Math.round(mealCost * 100) / 100,
        currency: currency,
        ingredients: scaledIngredients
      };
    }

    let slotCachedOptions = {};

    function buildInitialMeals() {
      const routine = ROUTINE_SPLITS[appState.routineKey];
      slotCachedOptions = {};

      for (const [slot, ratio] of Object.entries(routine)) {
        const slotKcal = appState.plan.target_calories * ratio;
        const baseSlot = slot.includes('snack') ? 'snack' : slot;
        const available = RECIPES_DB.filter(r => r.slot === baseSlot);

        const scaledOptions = available.map(r => scaleRecipe(r, slotKcal, appState.user.country));
        slotCachedOptions[slot] = scaledOptions;

        if (scaledOptions.length > 0 && !appState.selectedMeals[slot]) {
          appState.selectedMeals[slot] = scaledOptions[0]; // Auto-select first
        }
      }

      renderMeters();
      renderAllSlots();
      renderGroceryList();
    }

    // ==========================================
    // 4. RENDER METERS & BALANCES
    // ==========================================
    function renderMeters() {
      const plan = appState.plan;
      if (!plan) return;

      let loggedCal = 0, loggedPro = 0, loggedCarb = 0, loggedFat = 0;
      Object.values(appState.selectedMeals).forEach(m => {
        loggedCal += m.calories;
        loggedPro += m.protein;
        loggedCarb += m.carbs;
        loggedFat += m.fat;
      });

      const remCal = plan.target_calories - loggedCal;
      const remPro = Math.round((plan.target_protein - loggedPro) * 10) / 10;
      const remCarb = Math.round((plan.target_carbs - loggedCarb) * 10) / 10;
      const remFat = Math.round((plan.target_fat - loggedFat) * 10) / 10;
      const remWater = plan.water_target - appState.loggedWater;

      document.getElementById('cal-logged').innerText = loggedCal;
      document.getElementById('cal-target').innerText = plan.target_calories;
      document.getElementById('cal-remain-label').innerText = remCal >= 0
        ? `${remCal} kcal ${translations[currentLang].left}`
        : `${Math.abs(remCal)} kcal ${translations[currentLang].over}`;

      document.getElementById('pro-logged').innerText = Math.round(loggedPro);
      document.getElementById('pro-target').innerText = plan.target_protein;
      document.getElementById('pro-remain-label').innerText = remPro >= 0
        ? `${remPro}g ${translations[currentLang].left}`
        : `${Math.abs(remPro)}g ${translations[currentLang].over}`;

      document.getElementById('carb-logged').innerText = Math.round(loggedCarb);
      document.getElementById('carb-target').innerText = plan.target_carbs;
      document.getElementById('carb-remain-label').innerText = remCarb >= 0
        ? `${remCarb}g ${translations[currentLang].left}`
        : `${Math.abs(remCarb)}g ${translations[currentLang].over}`;

      document.getElementById('fat-logged').innerText = Math.round(loggedFat);
      document.getElementById('fat-target').innerText = plan.target_fat;
      document.getElementById('fat-remain-label').innerText = remFat >= 0
        ? `${remFat}g ${translations[currentLang].left}`
        : `${Math.abs(remFat)}g ${translations[currentLang].over}`;

      document.getElementById('water-logged').innerText = appState.loggedWater;
      document.getElementById('water-target').innerText = plan.water_target;
      document.getElementById('water-remain-label').innerText = remWater >= 0
        ? `${remWater} ml ${translations[currentLang].left}`
        : `${Math.abs(remWater)} ml ${translations[currentLang].over}`;

      // Progress bars
      document.getElementById('cal-bar').style.width = Math.min(100, (loggedCal / plan.target_calories) * 100) + '%';
      document.getElementById('pro-bar').style.width = Math.min(100, (loggedPro / plan.target_protein) * 100) + '%';
      document.getElementById('carb-bar').style.width = Math.min(100, (loggedCarb / plan.target_carbs) * 100) + '%';
      document.getElementById('fat-bar').style.width = Math.min(100, (loggedFat / plan.target_fat) * 100) + '%';
      document.getElementById('water-bar').style.width = Math.min(100, (appState.loggedWater / plan.water_target) * 100) + '%';

      document.getElementById('status-badge').innerText = `${plan.target_calories} kcal • BMR: ${plan.bmr} • TDEE: ${plan.tdee}`;
    }

    function logWater(val) {
      appState.loggedWater += val;
      renderMeters();
    }

    // ==========================================
    // 5. RENDER RECIPES IN SLOTS
    // ==========================================
    function renderAllSlots() {
      const container = document.getElementById('slots-container');
      container.innerHTML = '';
      const routine = ROUTINE_SPLITS[appState.routineKey];

      for (const [slot, ratio] of Object.entries(routine)) {
        const options = slotCachedOptions[slot] || [];
        const selected = appState.selectedMeals[slot];
        const slotTitle = translations[currentLang][slot] || slot;
        const slotKcal = Math.round(appState.plan.target_calories * ratio);

        const card = document.createElement('div');
        card.className = "bg-white rounded-2xl p-6 border border-slate-200 shadow-sm space-y-4";
        card.innerHTML = `
          <div class="flex items-center justify-between border-b border-slate-100 pb-3">
            <div>
              <span class="text-xs font-bold uppercase text-emerald-600">${slotTitle}</span>
              <h3 class="text-base font-bold text-slate-900">${slotKcal} kcal Target (${Math.round(ratio * 100)}%)</h3>
            </div>
            ${selected ? `<span class="px-2.5 py-1 rounded-full bg-emerald-100 text-emerald-800 text-xs font-bold">${translations[currentLang].selected_badge}: ${selected['name_' + currentLang]}</span>` : ''}
          </div>

          <div class="grid grid-cols-1 md:grid-cols-2 gap-4">
            ${options.map(r => {
              const isSelected = selected && selected.id === r.id;
              return `
                <div class="p-4 rounded-xl border ${isSelected ? 'border-emerald-500 bg-emerald-50/30 ring-2 ring-emerald-500/20' : 'border-slate-200 bg-slate-50'} flex flex-col justify-between space-y-3">
                  <div>
                    <div class="flex justify-between items-start">
                      <h4 class="font-bold text-slate-900 text-sm">${r['name_' + currentLang]}</h4>
                      <span class="text-xs font-bold text-emerald-700 bg-white px-2 py-0.5 rounded border border-emerald-200">${r.cost.toFixed(2)} ${r.currency}</span>
                    </div>
                    <div class="text-xs text-slate-500 mt-1 flex space-x-2 rtl:space-x-reverse font-medium">
                      <span>${r.calories} kcal</span>
                      <span>•</span>
                      <span>${r.protein}g P</span>
                      <span>•</span>
                      <span>${r.carbs}g C</span>
                      <span>•</span>
                      <span>${r.fat}g F</span>
                    </div>

                    <!-- 1g Granularity Ingredients List -->
                    <div class="mt-3 bg-white p-3 rounded-lg border border-slate-200/80 text-xs space-y-1">
                      <div class="font-bold text-slate-700 mb-1 text-[11px] uppercase tracking-wider">Gram Measurements:</div>
                      ${r.ingredients.map(ing => `
                        <div class="flex justify-between text-slate-600">
                          <span>${ing['name_' + currentLang]}</span>
                          <span class="font-bold text-slate-900">${ing.weight_g} ${currentLang === 'ar' ? 'غ' : 'g'}</span>
                        </div>
                      `).join('')}
                    </div>

                    <!-- Clean Method without numbers -->
                    <div class="mt-3 text-xs text-slate-600 bg-slate-100/70 p-2.5 rounded-lg">
                      <div class="font-bold text-slate-700 mb-1">${translations[currentLang].method_title}</div>
                      <p class="whitespace-pre-line text-slate-500">${r['method_' + currentLang]}</p>
                    </div>
                  </div>

                  <button onclick="selectMealForSlot('${slot}', ${r.id})" class="w-full py-2 text-xs font-bold rounded-lg transition-all ${isSelected ? 'bg-emerald-600 text-white' : 'bg-white border border-slate-300 text-slate-700 hover:bg-slate-100'}">
                    ${isSelected ? translations[currentLang].selected_badge : translations[currentLang].select_recipe}
                  </button>
                </div>
              `;
            }).join('')}
          </div>
        `;
        container.appendChild(card);
      }
    }

    function selectMealForSlot(slot, recipeId) {
      const selected = slotCachedOptions[slot].find(r => r.id === recipeId);
      appState.selectedMeals[slot] = selected;
      renderMeters();
      renderAllSlots();
      renderGroceryList();
    }

    // ==========================================
    // 6. GROCERY LIST (ORGANIZED BY AISLE)
    // ==========================================
    function renderGroceryList() {
      const container = document.getElementById('grocery-departments-container');
      container.innerHTML = '';

      let aggregated = {};
      let totalCost = 0;
      let currency = REGIONAL_PRICING[appState.user.country].currency;

      Object.values(appState.selectedMeals).forEach(meal => {
        meal.ingredients.forEach(ing => {
          if (!aggregated[ing.id]) {
            aggregated[ing.id] = {
              name_en: ing.name_en,
              name_ar: ing.name_ar,
              dept_en: ing.dept_en,
              dept_ar: ing.dept_ar,
              weight_g: 0,
              cost: 0
            };
          }
          aggregated[ing.id].weight_g += ing.weight_g;
          aggregated[ing.id].cost += ing.cost;
          totalCost += ing.cost;
        });
      });

      document.getElementById('grocery-total-cost').innerText = `${totalCost.toFixed(2)} ${currency}`;

      // Group by Department
      const byDept = {};
      Object.values(aggregated).forEach(item => {
        const deptKey = item['dept_' + currentLang];
        if (!byDept[deptKey]) byDept[deptKey] = [];
        byDept[deptKey].push(item);
      });

      for (const [dept, items] of Object.entries(byDept)) {
        const block = document.createElement('div');
        block.className = "border border-slate-200 rounded-xl overflow-hidden";
        block.innerHTML = `
          <div class="bg-slate-100 px-4 py-2 font-bold text-xs uppercase tracking-wider text-slate-700">${dept}</div>
          <table class="w-full text-sm text-slate-700">
            <tbody class="divide-y divide-slate-100">
              ${items.map(item => `
                <tr class="hover:bg-slate-50 transition-colors">
                  <td class="py-2.5 px-4 w-12 text-left rtl:text-right">
                    <input type="checkbox" onchange="togglePantry(this)" class="w-4 h-4 text-emerald-600 rounded border-slate-300 focus:ring-emerald-500">
                  </td>
                  <td class="py-2.5 px-4 font-semibold text-slate-900 item-name">${item['name_' + currentLang]}</td>
                  <td class="py-2.5 px-4 text-center font-bold text-slate-800">${item.weight_g} ${currentLang === 'ar' ? 'غ' : 'g'}</td>
                  <td class="py-2.5 px-4 text-right rtl:text-left font-bold text-emerald-700">${item.cost.toFixed(2)} ${currency}</td>
                </tr>
              `).join('')}
            </tbody>
          </table>
        `;
        container.appendChild(block);
      }
    }

    function togglePantry(cb) {
      const name = cb.closest('tr').querySelector('.item-name');
      if (cb.checked) {
        name.classList.add('line-through', 'text-slate-400');
      } else {
        name.classList.remove('line-through', 'text-slate-400');
      }
    }
  </script>
</body>
</html>
"""

# Embed and render directly inside Colab
display(
    HTML(f"""
<div style="border: 1px solid #e2e8f0; border-radius: 12px; overflow: hidden; max-width: 100%; box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.1);">
  <iframe srcdoc='{APP_HTML.replace("'", "&#39;")}'
          width="100%"
          height="880px"
          frameborder="0"
          style="background: #f8fafc;">
  </iframe>
</div>
""")
)